# Holo4-35B-A3B on Kaggle: OpenAI-compatible API + ngrok

Serves H Company's Holo4-35B-A3B (35B MoE, 3B active, vision + tool calling) as a 4-bit GGUF
through `llama.cpp`'s `llama-server`. It exposes OpenAI-style endpoints
(`/v1/chat/completions`, `/v1/completions`, `/v1/models`, `/v1/embeddings`) and tunnels them out with ngrok.

**Setup before running**
1. Session options → Accelerator: **GPU T4 x2** (about 30 GB VRAM total). A single 16 GB GPU needs `N_CPU_MOE` raised in cell 5.
2. Session options → Internet: **On**.
3. Add-ons → Secrets: `NGROK_AUTHTOKEN`, `NGROK_DOMAIN` (e.g. `my-name.ngrok-free.app`, no `https://`).
   Optional: `HF_TOKEN` (faster downloads), `API_KEY` (protects your endpoint with a bearer key).

**Why GGUF / llama.cpp?** T4 and P100 GPUs have no bf16/FP8/FP4 support, so FP8 and NVFP4 checkpoints
are not usable. Q4_K_M is about 20 GB, which fits across 2x T4 with room for KV cache and the vision projector.

## 1. Check GPUs

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv

## 2. Install Python deps

In [ ]:
!pip install -q pyngrok huggingface_hub openai pillow requests

## 3. Build llama.cpp with CUDA (about 10 min the first time; skipped if already built)

In [ ]:
import subprocess, os, shutil, textwrap

BUILD_SH = r'''
set -o pipefail
export PATH=/usr/local/cuda/bin:$PATH
export CUDACXX=$(which nvcc || echo /usr/local/cuda/bin/nvcc)
cd /kaggle/working
if [ -x llama.cpp/build/bin/llama-server ]; then echo "already built"; exit 0; fi

echo "== internet check =="; git ls-remote https://github.com/ggml-org/llama.cpp HEAD || { echo "NO INTERNET: enable Internet in session options"; exit 11; }
echo "== nvcc =="; $CUDACXX --version || { echo "nvcc not found"; exit 12; }

CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader 2>/dev/null | head -1 | tr -d '. \r')
CC=${CC:-75}
echo "CUDA arch: $CC"

rm -rf llama.cpp
git clone --depth 1 https://github.com/ggml-org/llama.cpp || exit 13
cd llama.cpp
cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=$CC -DCMAKE_CUDA_COMPILER=$CUDACXX \
      -DCMAKE_BUILD_TYPE=Release -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF 2>&1 | tail -40 || exit 14
cmake --build build --config Release -j $(nproc) --target llama-server 2>&1 | tail -60 || exit 15
ls -la build/bin/llama-server
'''
open("/kaggle/working/build.sh", "w").write(BUILD_SH)
r = subprocess.run(["bash", "/kaggle/working/build.sh"], capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-3000:])
print("exit code:", r.returncode)
if r.returncode != 0 or not os.path.exists("/kaggle/working/llama.cpp/build/bin/llama-server"):
    raise RuntimeError("llama.cpp build failed, send the output above")

## 4. Load secrets and download the quantized model

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def get_secret(name, required=False):
    try:
        return secrets.get_secret(name)
    except Exception:
        if required:
            raise RuntimeError(f"Missing Kaggle secret: {name}")
        return None

NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN", required=True)
NGROK_DOMAIN = get_secret("NGROK_DOMAIN", required=True).replace("https://", "").replace("http://", "").strip("/ ")
HF_TOKEN = get_secret("HF_TOKEN")
API_KEY = get_secret("API_KEY")  # optional; if set, clients must send "Authorization: Bearer <API_KEY>"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# ---- Model config ----
REPO = "Hcompany/Holo4-35B-A3B-GGUF"   # official GGUF repo
QUANT = "Q4_K_M"                       # ~20 GB. Use "Q4_0" (~18 GB) if you need more KV headroom
MODEL_ALIAS = "holo4-35b-a3b"          # model name clients will use

from huggingface_hub import list_repo_files, hf_hub_download

files = list_repo_files(REPO)
print("\n".join(f for f in files if f.endswith(".gguf")))

weights = sorted(f for f in files if f.endswith(".gguf")
                 and QUANT.lower() in f.lower() and "mmproj" not in f.lower())
if not weights:
    raise RuntimeError(f"No {QUANT} GGUF found in {REPO}; pick a quant from the list above.")

local_weights = [hf_hub_download(REPO, f, local_dir="/kaggle/working/models") for f in weights]
MODEL_PATH = local_weights[0]   # for sharded files llama.cpp finds the rest from the first shard

mmprojs = sorted(f for f in files if "mmproj" in f.lower() and f.endswith(".gguf"))
MMPROJ_PATH = None
if mmprojs:
    pick = next((f for f in mmprojs if "f16" in f.lower()), mmprojs[0])
    MMPROJ_PATH = hf_hub_download(REPO, pick, local_dir="/kaggle/working/models")
else:
    print("WARNING: no mmproj file in repo, vision input will be disabled.")

print("Model :", MODEL_PATH)
print("mmproj:", MMPROJ_PATH)

## 5. Start the OpenAI-compatible server

In [ ]:
import subprocess, time, requests, shlex

PORT = 8000
CTX = 32768          # total context; split across parallel slots
PARALLEL = 2         # concurrent requests (each gets CTX/PARALLEL tokens)
KV_QUANT = "q8_0"    # KV cache quantization (q8_0 halves KV memory; use "f16" if you see problems)
N_CPU_MOE = 0        # keep the experts of N layers in system RAM. 0 on 2x T4. Try 10-20 on a single 16 GB GPU

cmd = [
    "/kaggle/working/llama.cpp/build/bin/llama-server",
    "-m", MODEL_PATH,
    "--alias", MODEL_ALIAS,
    "--host", "0.0.0.0", "--port", str(PORT),
    "-ngl", "999",
    "--split-mode", "layer",
    "-c", str(CTX),
    "-np", str(PARALLEL),
    "-ctk", KV_QUANT, "-ctv", KV_QUANT,
    "--jinja",                       # model's own chat template (tool calling + vision messages)
]
if MMPROJ_PATH:
    cmd += ["--mmproj", MMPROJ_PATH]
if N_CPU_MOE:
    cmd += ["--n-cpu-moe", str(N_CPU_MOE)]
if API_KEY:
    cmd += ["--api-key", API_KEY]

LOG = "/kaggle/working/llama-server.log"
subprocess.run(["pkill", "-f", "llama-server"], check=False)
time.sleep(2)
server = subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("Launched:", " ".join(shlex.quote(c) for c in cmd if c != API_KEY))

# wait for the model to load
t0 = time.time()
while True:
    if server.poll() is not None:
        print(open(LOG).read()[-4000:])
        raise RuntimeError("llama-server exited, see log above")
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    if time.time() - t0 > 1200:
        print(open(LOG).read()[-4000:])
        raise TimeoutError("Server did not become healthy in 20 min")
    time.sleep(5)
print(f"Server ready after {time.time()-t0:.0f}s")
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

## 6. Open the ngrok tunnel on your reserved domain

In [ ]:
from pyngrok import ngrok

ngrok.kill()
ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(addr=PORT, proto="http", domain=NGROK_DOMAIN)
BASE_URL = f"https://{NGROK_DOMAIN}/v1"
print("Public base URL:", BASE_URL)
print("Model name     :", MODEL_ALIAS)
print("API key        :", "required (your API_KEY secret)" if API_KEY else "not required (set an API_KEY secret to protect it)")

## 7. Smoke test (text + vision + streaming) through the public URL

In [ ]:
import base64, io
from openai import OpenAI
from PIL import Image, ImageDraw

client = OpenAI(base_url=BASE_URL, api_key=API_KEY or "sk-none")

print(client.models.list().data[0].id)

# text
r = client.chat.completions.create(
    model=MODEL_ALIAS,
    messages=[{"role": "user", "content": "Say hello in one short sentence."}],
    max_tokens=64,
)
print(r.choices[0].message.content)

# vision (draws a button and asks where it is)
img = Image.new("RGB", (640, 360), "white")
d = ImageDraw.Draw(img)
d.rectangle([420, 250, 560, 310], fill="royalblue")
d.text((460, 272), "Submit", fill="white")
buf = io.BytesIO(); img.save(buf, format="PNG")
b64 = base64.b64encode(buf.getvalue()).decode()

r = client.chat.completions.create(
    model=MODEL_ALIAS,
    messages=[{"role": "user", "content": [
        {"type": "text", "text": "Describe the UI element in this screenshot and roughly where it is."},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
    ]}],
    max_tokens=200,
)
print(r.choices[0].message.content)

# streaming
for chunk in client.chat.completions.create(
    model=MODEL_ALIAS, stream=True, max_tokens=60,
    messages=[{"role": "user", "content": "Count from 1 to 5."}],
):
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

## 8. Keep the session alive

Leave this cell running; Kaggle stops the server when the notebook session ends (max about 9-12 h per session).
Use the endpoint from anywhere:

```python
from openai import OpenAI
client = OpenAI(base_url="https://YOUR_DOMAIN/v1", api_key="YOUR_API_KEY_OR_ANYTHING")
client.chat.completions.create(model="holo4-35b-a3b", messages=[{"role": "user", "content": "hi"}])
```
```bash
curl https://YOUR_DOMAIN/v1/chat/completions \
  -H "Authorization: Bearer YOUR_API_KEY" -H "Content-Type: application/json" \
  -d '{"model":"holo4-35b-a3b","messages":[{"role":"user","content":"hi"}]}'
```

In [ ]:
import time
print("Serving at", BASE_URL, "(interrupt the cell to stop)")
try:
    while True:
        if server.poll() is not None:
            print("llama-server died. Last log lines:\n", open(LOG).read()[-3000:])
            break
        time.sleep(60)
except KeyboardInterrupt:
    pass
finally:
    ngrok.kill()
    server.terminate()